# Landsat

Use the launch button at the top of this page to open the full notebook in Google Colab. The workflow runs on Earth Engine and keeps the imagery in the cloud.


In [ ]:
# %pip install earthengine-api
# %pip install geemap

In [ ]:
import ee
import geemap

In [ ]:
ee.Authenticate()              # Step 1: authenticate your Earth Engine account


In [ ]:
ee.Initialize(project='your-cloud-project')    # Step 2: initialize Earth Engine with your Cloud project ID


## Hands-on workflow


### Task 1: Create a map with geemap


In [ ]:
geemap.Map()

In [ ]:
Map = geemap.Map()    # center = [lat, lon]

In [ ]:
Map

In [ ]:
Map.add_basemap("Esri.WorldImagery")     # Add Esri World Imagery as the basemap


### Task 2: Define a square ROI


In [ ]:
# A fixed square ROI around Los Angeles, California.
roi = ee.Geometry.Rectangle([-118.75, 33.70, -117.75, 34.70])

Map.centerObject(roi, 9)
Map.addLayer(roi, {'color': 'red'}, 'ROI')
Map

## Landsat mission overview


![Landsat notebook figure 1](../images/gee/landsat-01.png)

Landsat-based global surface change timelapse (1984-2022): https://earthengine.google.com/timelapse/


### Landsat mission summary

| Satellite | Launch year | Main sensor(s) | Bands | Spectral range | Spatial resolution | Notes |
|---|---:|---|---:|---|---|---|
| Landsat 1 | 1972 | MSS | 4 | 500-1100 nm | 60 m | Visible and near-infrared foundation bands |
| Landsat 2 | 1975 | MSS | 4 | Same as Landsat 1 | 60 m | Similar to Landsat 1 |
| Landsat 3 | 1978 | MSS + experimental TIR | 5 | 500-1200 nm, including thermal infrared | 60 m (MSS) | Experimental thermal infrared band |
| Landsat 4 | 1982 | MSS + TM | 7 | 450-1250 nm + thermal infrared | 30 m (TM) | Added blue, mid-infrared, and thermal bands |
| Landsat 5 | 1984 | MSS (early) + TM | 7 | Same as Landsat 4 | 30 m (TM) | Operated until 2013; the longest-lived Landsat satellite |
| Landsat 6 | 1993 | ETM | - | - | - | Launch failed; no operational imagery |
| Landsat 7 | 1999 | ETM+ | 8 | 450-1250 nm + thermal infrared | 30 m + 15 m panchromatic | Added a 15 m panchromatic band |
| Landsat 8 | 2013 | OLI + TIRS | 11 | 430-1250 nm + thermal infrared | 30 m + 15 m panchromatic | New sensors with improved radiometric precision |
| Landsat 9 | 2021 | OLI-2 + TIRS-2 | 11 | Same as Landsat 8 | 30 m + 15 m panchromatic | Similar to Landsat 8, with continuity and redundancy |


### Common Landsat spectral bands

| Band | Band name | Wavelength (nm) | L1-3 (MSS) | L4 (TM) | L5 (TM) | L7 (ETM+) | L8 (OLI/TIRS) | L9 (OLI-2/TIRS-2) |
|---:|---|---|---|---|---|---|---|---|
| 1 | Coastal aerosol | 433-453 | - | - | - | - | Yes | Yes |
| 2 | Blue | 450-515 | - | Yes | Yes | Yes | Yes | Yes |
| 3 | Green | 525-600, or 500-600 | MSS B4 | Yes | Yes | Yes | Yes | Yes |
| 4 | Red | 630-680, or 600-700 | MSS B5 | Yes | Yes | Yes | Yes | Yes |
| 5 | Near infrared (NIR) | 760-900, or 700-1100 | MSS B6 | Yes | Yes | Yes | Yes | Yes |
| 6 | SWIR 1 | 1560-1660 | - | Yes | Yes | Yes | Yes | Yes |
| 7 | SWIR 2 | 2100-2300 | - | Yes | Yes | Yes | Yes | Yes |
| 8 | Panchromatic | 500-900 (L7) / 500-680 (L8/9) | - | - | - | Yes | Yes | Yes |
| 9 | Cirrus | 1360-1390 | - | - | - | - | Yes | Yes |
| 10 | Thermal infrared 1 (TIR 1) | 10600-11190 | - | Band 6 | Band 6 | Band 6 | Yes | Yes |
| 11 | Thermal infrared 2 (TIR 2) | 11500-12510 | - | - | - | - | Yes | Yes |


### Task 3: Find Landsat images covering the ROI in June 2025


https://developers.google.com/earth-engine/datasets/catalog/landsat

In [ ]:
images_20250522 = ee.ImageCollection("LANDSAT/LC08/C02/T2_L2").filterBounds(roi).filterDate('2025-05-01', '2025-07-01')

In [ ]:
ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).filterDate('2024-07-01', '2024-08-01')

In [ ]:
images_202407 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).filterDate('2024-07-01', '2024-08-01')

In [ ]:
vis_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
}

In [ ]:
Map.addLayer(images_20250522, vis_params, '20250522')

In [ ]:
Map

In [ ]:
Map

### Task 4: Mask cloudy pixels


In [ ]:
a_image = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).filterDate('2024-07-01', '2024-08-01').first()

In [ ]:
a_image

In [ ]:
images_202407 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).filterDate('2024-07-01', '2024-08-01')

In [ ]:
images_202407

In [ ]:
images_202407.toList(3).get(2)

In [ ]:
a_image = ee.Image(images_202407.toList(3).get(2))

In [ ]:
Map.addLayer(a_image, {'bands': ['SR_B4', 'SR_B3', 'SR_B2']}, '20240729')

In [ ]:
Map

![Landsat notebook figure 4](../images/gee/landsat-04-en.png)

In [ ]:
a_image.select('QA_PIXEL').bitwiseAnd(0b11111).eq(0)
# a_image.select('QA_PIXEL').bitwiseAnd(31).eq(0)

# 0b11111 = 0000 0000 0001 1111 = 2^4 + 2^3 + 2^2 + 2^1 + 2^0 = 31

# Bitwise operation walkthrough
# QA_PIXEL value: 0011 0101 0000 1010
#          mask: 0000 0000 0001 1111
#        result: 0000 0000 0000 1010

In [ ]:
qa_mask = a_image.select('QA_PIXEL').bitwiseAnd(0b11111).eq(0)

In [ ]:
qa_mask

In [ ]:
Map.addLayer(qa_mask, {'min':0, 'max':1, 'palette':['red', 'green']}, 'qa_mask')

In [ ]:
Map

In [ ]:
a_image_no_cloud = a_image.updateMask(qa_mask)

In [ ]:
Map.addLayer(a_image_no_cloud, {'bands': ['SR_B4', 'SR_B3', 'SR_B2']}, '20240729 no cloud')

In [ ]:
Map

### Task 5: Composite the 2024 Landsat images into one image


In [ ]:
ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(roi).filterDate('2024-01-01', '2024-12-31')

In [ ]:
found_images_2024 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(roi).filterDate('2024-01-01', '2024-12-31')

In [ ]:
# Define a function to mask cloudy pixels.
def mask_cloud(image):
  qa_mask = image.select('QA_PIXEL').bitwiseAnd(0b11111).eq(0)
  return image.updateMask(qa_mask)

In [ ]:
found_images_2024.map(mask_cloud)

In [ ]:
found_images_2024_no_cloud = found_images_2024.map(mask_cloud)

In [ ]:
composite_image_2024 = found_images_2024_no_cloud.median()

In [ ]:
Map.addLayer(composite_image_2024, {'bands': ['SR_B4', 'SR_B3', 'SR_B2']}, '2024')

In [ ]:
Map

### Task 6: Convert raw pixel values to surface reflectance with scale factors


Landsat bands in Google Earth Engine use scale factors to convert raw digital numbers (DN) into physically meaningful values such as surface reflectance or temperature. These scale factors and offsets are used because the original data are stored as scaled integers to reduce storage size and improve processing efficiency.


In [ ]:
# Regex note: '.' matches one character, so this selects SR_B1 through SR_B7.
optical_bands = composite_image_2024.select('SR_B.').multiply(0.0000275).add(-0.2)

In [ ]:
# Regex note: '.*' matches zero or more characters, so this selects thermal bands such as ST_B10.
thermal_bands = composite_image_2024.select('ST_B.*').multiply(0.00341802).add(149.0)

In [ ]:
optical_bands

In [ ]:
thermal_bands

In [ ]:
# Replace the original bands with the scaled bands.
processed_image = composite_image_2024.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)    # Image.addBands(srcImg, names, overwrite=False)

In [ ]:
Map.addLayer(processed_image, {'bands': ['SR_B4', 'SR_B3', 'SR_B2']}, '2024 (converted)')

In [ ]:
Map

### Task 7: Build annual Landsat composites from 1985 to 2024


In [ ]:
found_images_2024 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(roi).filterDate('2024-01-01', '2024-12-31')

In [ ]:
ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

In [ ]:
ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

In [ ]:
ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")

In [ ]:
ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")

In [ ]:
found_images_2024

In [ ]:
found_images_2023 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(roi).filterDate('2023-01-01', '2023-12-31')

In [ ]:
found_images_2022 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(roi).filterDate('2022-01-01', '2022-12-31')

In [ ]:
names = locals()

In [ ]:
for t in range(2014, 2022):
  names[f'found_images_{t}'] = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).filterDate(f'{t}-01-01', f'{t}-12-31')

In [ ]:
for t in range(2000, 2014):
  names[f'found_images_{t}'] = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").filterBounds(roi).filterDate(f'{t}-01-01', f'{t}-12-31')

In [ ]:
for t in range(1985, 2000):
  names[f'found_images_{t}'] = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").filterBounds(roi).filterDate(f'{t}-01-01', f'{t}-12-31')

In [ ]:
# Define a function to mask clouds and apply band scaling.
def mask_landsat(image):
  qa_mask = image.select('QA_PIXEL').bitwiseAnd(0b11111).eq(0)
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

  return image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True).updateMask(qa_mask)

In [ ]:
processed_images_2024 = found_images_2024.map(mask_landsat)

In [ ]:
composite_image_2024 = processed_images_2024.median()

In [ ]:
for t in range(1985, 2025):
  names[f'processed_images_{t}'] = names[f'found_images_{t}'].map(mask_landsat)
  names[f'composite_image_{t}'] = names[f'processed_images_{t}'].median()

In [ ]:
Map.addLayer(composite_image_1985, {'bands': ['SR_B3', 'SR_B2', 'SR_B1'], 'min': 0, 'max': 0.3}, '1985')

In [ ]:
Map.addLayer(composite_image_2010, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}, '2010')

In [ ]:
Map

### Task 8: Export Landsat data


1. Clip the image to the region of interest


In [ ]:
composite_image_2024.clip(roi)

In [ ]:
clipped_tile = composite_image_2024.clip(roi)

2. Export the image to Google Drive with native Earth Engine export tools


In [ ]:
task = ee.batch.Export.image.toDrive(
    image=clipped_tile,
    description='export_clipped_tile',
    folder='GEE_exports',
    fileNamePrefix='clipped_tile',
    region=roi,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e13
)

In [ ]:
task.start()

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=clipped_tile,
    description='export_clipped_tile',
    folder='GEE_exports',
    fileNamePrefix='clipped_tile',
    region=roi,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e13
)

In [ ]:
task.start()

3. Export the image to the local Colab runtime with geemap


In [ ]:
geemap.ee_export_image(
    clipped_tile,
    filename='clipped_tile.tif',
    scale=30,
    region=roi,
    file_per_band=False,
)

In [ ]:
geemap.ee_export_image(
    clipped_tile,
    filename='clipped_tile.tif',           # Local output filename
    scale=30,                              # Landsat optical bands are typically 30 m; otherwise Earth Engine may use a coarse default scale.
    # region=roi,                          # The region should be a geometry.
    file_per_band=False,                   # Save all bands into one file.
)

In [ ]:
clipped_tile_rgb = clipped_tile.select(['SR_B4', 'SR_B3', 'SR_B2'])

In [ ]:
geemap.ee_export_image(
    clipped_tile_rgb,
    filename='clipped_tile_rgb.tif',
    scale=30,
    # region=roi,
    file_per_band=False,
)

In [ ]:
Map.addLayer(clipped_tile_rgb, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}, 'gg')

In [ ]:
Map

In [ ]:
# Stretch 0-1 reflectance values to 0-255.
rgb_vis = clipped_tile_rgb.multiply(255).uint8()

In [ ]:
# Export again.
geemap.ee_export_image(
    rgb_vis,
    filename='clipped_tile_rgb_vis.tif',
    scale=30,
    region=roi,
    file_per_band=False,
)

### Task 9: Export 1985-2024 Landsat composites as a GIF


1. Rename the bands so the workflow is consistent across missions


In [ ]:
image_list = [names[f'composite_image_{t}'].select(['SR_B5', 'SR_B4', 'SR_B3', 'SR_B2', 'SR_B1'], ['SWIR', 'NIR', 'R', 'G', 'B']) for t in range(1985, 2014)] \
            + [names[f'composite_image_{t}'].select(['SR_B6', 'SR_B5', 'SR_B4', 'SR_B3', 'SR_B2'], ['SWIR', 'NIR', 'R', 'G', 'B']) for t in range(2014, 2025)]

2. Convert the image list to an `ImageCollection`


In [ ]:
image_collection = ee.ImageCollection(image_list)

3. Set the GIF visualization parameters


In [ ]:
video_args = {
    'dimensions': 600,  # Alternatively use 'width': 720, 'height': 480
    'region': roi,      # Your study area
    'framesPerSecond': 2,
    'bands': ['R', 'G', 'B'],
    'min': 0.0,
    'max': 0.3,
    # 'gamma': 1.3
}

4. Export the GIF with geemap


In [ ]:
geemap.download_ee_video(
    image_collection,
    video_args,
    out_gif='landsat_1985_2024_.gif',
)

5. Add year labels to the GIF


In [ ]:
year_labels = [t for t in range(1985, 2025)]
print(year_labels)

In [ ]:
geemap.add_text_to_gif(
    'landsat_1985_2024_.gif',
    'landsat_1985_2024.gif',
    xy=("3%", "3%"),
    text_sequence=year_labels,
    font_size=30,
    font_color="#ffffff",
    duration=500,
)

### Task 10: Compute NDVI and export 1985-2024 images as a GIF


In [ ]:
# Create the NDVI image list.
ndvi_list_part1 = [names[f'composite_image_{t}'].normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI') for t in range(1985, 2014)]     # for Landsat 5/7
ndvi_list_part2 = [names[f'composite_image_{t}'].normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI') for t in range(2014, 2025)]     # for Landsat 8/9
ndvi_list = ndvi_list_part1 + ndvi_list_part2

In [ ]:
# Convert to an ImageCollection.
ndvi_collection = ee.ImageCollection(ndvi_list)

In [ ]:
# Set export parameters.
video_args_ndvi = {
    'dimensions': 600,
    'region': roi,
    'framesPerSecond': 2,
    'bands': ['NDVI'],
    'min': 0.0,
    'max': 0.8,
    'palette': ['white', 'green'],
}

In [ ]:
geemap.download_ee_video(
    ndvi_collection,
    video_args_ndvi,
    out_gif='landsat_1985_2024_NDVI_.gif'
)

In [ ]:
geemap.add_text_to_gif(
    'landsat_1985_2024_NDVI_.gif',
    'landsat_1985_2024_NDVI.gif',
    xy=("3%", "3%"),
    text_sequence=text,
    font_size=30,
    font_color="red",
    duration=500,
)

In [ ]:
  geemap.add_text_to_gif(
    'landsat_1985_2024_NDVI.gif',
    'landsat_1985_2024_NDVI.gif',
    xy=("53%", "3%"),
    text_sequence='Nova Ubiratã, Brazil',
    font_size=30,
    font_color="red",
    duration=500,
)

### Task 11: Compute NDBI and export 1985-2024 images as a GIF


In [ ]:
ndbi_list = [
    names[f'composite_image_{t}'].normalizedDifference(['SR_B5', 'SR_B4']).rename('NDBI')
    for t in range(1985, 2014)
] + [
    names[f'composite_image_{t}'].normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    for t in range(2014, 2025)
]

In [ ]:
# Convert to an ImageCollection.
ndbi_collection = ee.ImageCollection(ndbi_list)

In [ ]:
video_args_ndbi = {
    'dimensions': 600,
    'region': roi,
    'framesPerSecond': 2,
    'bands': ['NDBI'],
    'min': -0.5,
    'max': 0.5,
    'palette': ['#FFFFFF', '#AAAAAA', '#FFAA00', '#FF0000']
}

In [ ]:
geemap.download_ee_video(
    ndbi_collection,
    video_args_ndbi,
    out_gif='landsat_1985_2024_NDBI_.gif'
)

In [ ]:
geemap.add_text_to_gif(
    'landsat_1985_2024_NDBI_.gif',
    'landsat_1985_2024_NDBI.gif',
    xy=("3%", "3%"),
    text_sequence=text,
    font_size=30,
    font_color="black",
    duration=500,
)

In [ ]:
geemap.add_text_to_gif(
    'landsat_1985_2024_NDBI.gif',
    'landsat_1985_2024_NDBI.gif',
    xy=("53%", "3%"),
    text_sequence='Nova Ubiratã, Brazil',
    font_size=30,
    font_color="black",
    duration=500,
)

## References

- Google Earth Engine Data Catalog. (n.d.). [USGS Landsat 8 Level 2, Collection 2, Tier 1](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2).
- U.S. Geological Survey. (n.d.). [Landsat Collection 2](https://www.usgs.gov/landsat-missions/landsat-collection-2).
